In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl
import numpy as np

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results

results_dir = PROJECT_ROOT / "results"

## DRG classification results

In [2]:
def process_drg_prediction_results(input_dir, **kwargs) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual"),
        expected_expr=pl.col("expected"),
        filter_ambiguous=pl.col("stop_reason") == Reason.GOT_TOKEN,
        drg=True,
        **kwargs,
    )

In [3]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.mimic_icu import DrgPredictionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = DrgPredictionDataset(path)
len(dataset)

46685

In [16]:
def top_k_accuracy(k, probs, codes, expected):
    top_k_indices = np.argpartition(
        probs, -k, axis=1
    )[:, -k:]

    top_k_codes = codes[top_k_indices]

    return np.mean(
        np.any(top_k_codes == expected[:, None], axis=1)
    )

In [17]:
for input_dir in (results_dir / Task.DRG_PREDICTION).iterdir():
    df = process_drg_prediction_results(input_dir)


    prob_cols = [c for c in df.columns if c.startswith("DRG//")]
    probs = df.select(prob_cols).to_numpy()
    expected = df["expected"].to_numpy()

    codes = np.array(prob_cols)

    for k in [1, 2, 3, 5]:
        print(f"Top-{k}: {top_k_accuracy(k, probs, codes, expected):.3f}")
    

Top-1: 0.645
Top-2: 0.744
Top-3: 0.786
Top-5: 0.830
